DeepEval Agentic Metrics Evaluation

This project validates an AI agent workflow using DeepEval agentic metrics.

An agent usually does:
user request
→ plans or decides steps
→ selects tools
→ calls tools
→ uses tool results
→ gives final answer

Agentic metrics check:
- Did the agent complete the task?
- Did the agent choose the correct tool?
- Did the agent pass correct arguments?
- Did the agent follow the plan?
- Did the agent avoid unnecessary steps?

This project evaluates AI agent behavior using DeepEval agentic metrics.

In [1]:
!pip install -U deepeval groq langchain langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.8/662.8 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.29.0 requires cli

!pip install deepeval==4.0.4 groq langchain langchain-groq -q

!pip install -q click==8.3.3

In [2]:
import os
from google.colab import userdata
from groq import Groq
import json

from deepeval.models import DeepEvalBaseLLM

In [3]:
def make_groq_strict(schema):
    if isinstance(schema, dict):
        if schema.get("type") == "object":
            schema["additionalProperties"] = False

            if "properties" in schema:
                schema["required"] = list(schema["properties"].keys())

        for value in schema.values():
            make_groq_strict(value)

    elif isinstance(schema, list):
        for value in schema:
            make_groq_strict(value)

    return schema

In [4]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


class GroqDeepEvalModel(DeepEvalBaseLLM):

    def __init__(self, model_name="openai/gpt-oss-20b"):
    #def __init__(self, model_name="allam-2-7b"):
        self.model_name = model_name
        self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def load_model(self):
        return self.client
    """
    def generate(self, prompt: str, **kwargs) -> str:
        schema = kwargs.get("schema")

        response_format = {
            "type": "json_schema",
            "json_schema": {
                "name": schema.__name__.lower(),
                "strict": True,
                "schema": make_groq_strict(
                    schema.model_json_schema()
                )
            }
        }
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=256,
            response_format=response_format,
            reasoning_format="hidden"
        )

        return response.choices[0].message.content
    """
    def generate(self, prompt: str, **kwargs) -> str:
        print("Prompt characters:", len(prompt))
        print("Approximate prompt tokens:", len(prompt) // 4)
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "Return compact JSON. Keep the task and outcome brief."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=512,
            include_reasoning=False
        )

        raw_response = response.choices[0].message.content or ""


        start = raw_response.find("{")
        end = raw_response.rfind("}")

        if start == -1 or end == -1:
            raise ValueError(f"Groq returned this raw content: {raw_response!r}")

        parsed_response = json.loads(
            raw_response[start:end + 1]
        )

        return json.dumps(parsed_response)
    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)

    def get_model_name(self):
        return self.model_name


groq_model = GroqDeepEvalModel()

print("Groq evaluator connected.")

Groq evaluator connected.


In [5]:
from google.colab import files

uploaded = files.upload()

Saving agent_working_2.py to agent_working_2.py


In [7]:
!pip install langchain-groq -q

In [6]:
from agent_working_2 import support_agent

print("Real agent imported successfully.")

LangChain tools registered: 20
LangChain Groq agent created.
Real agent imported successfully.


In [8]:
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.evaluate import ErrorConfig, AsyncConfig
from deepeval.tracing import observe, update_current_trace
from deepeval.metrics import TaskCompletionMetric, StepEfficiencyMetric, ToolCorrectnessMetric, ArgumentCorrectnessMetric, PlanQualityMetric, PlanAdherenceMetric

In [9]:
tool_correctness_metric = ToolCorrectnessMetric(
    model=groq_model
)

print("Tool Correctness metric created.")

Tool Correctness metric created.


In [10]:
import pandas as pd

agent_df = pd.read_csv("agentic_metrics_dataset.csv")

print("Agentic metrics dataset loaded.")
print("Total test cases:", len(agent_df))

Agentic metrics dataset loaded.
Total test cases: 35


In [11]:
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.test_case import ToolCall

agent_goldens = []

for _, row in agent_df.iterrows():
    tool_name = str(row["expected_tools"]).strip()
    expected_tools = (
        []
        if tool_name == "" or tool_name.lower() == "nan"
        else [ToolCall(name=tool_name)]
    )

    agent_goldens.append(
        Golden(
            input=row["user_task"],
            expected_output=row["expected_outcome"],
            expected_tools=expected_tools,
        )
    )

agent_dataset = EvaluationDataset(goldens=agent_goldens)

print("DeepEval dataset created.")
print("Total test cases:", len(agent_goldens))

DeepEval dataset created.
Total test cases: 35


In [12]:
task_completion_metric = TaskCompletionMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Task Completion metric created.")

Task Completion metric created.


In [13]:
from deepeval.tracing import observe, update_current_span
from deepeval.test_case import LLMTestCase
from agent_working_2 import support_agent as _support_agent

@observe(name="support_agent_evaluation")
def traced_support_agent(user_input):
    return _support_agent(user_input)

@observe(name="task_completion_evaluation", metrics=[task_completion_metric])
def evaluate_task_completion(user_task, answer, expected_outcome):
    update_current_span(
        test_case=LLMTestCase(
            input=user_task,
            actual_output=answer,
            expected_output=expected_outcome,
        )
    )

@observe(name="tool_correctness_evaluation", metrics=[tool_correctness_metric])
def evaluate_tool_correctness(user_task, answer, actual_tools, expected_tools):
    update_current_span(
        test_case=LLMTestCase(
            input=user_task,
            actual_output=answer,
            tools_called=actual_tools or [],
            expected_tools=expected_tools or [],
        )
    )

def run_agentic_evaluation_case(golden):
    answer = traced_support_agent(golden.input)
    actual_tools = getattr(_support_agent, "last_tools_called", []) or []

    return answer, actual_tools

print("Metric-specific evaluation wrappers created.")

Metric-specific evaluation wrappers created.


In [14]:
import time

In [15]:
from deepeval import evaluate

evaluation_results = []

def collect_metric_data(value, test_case_number, path="trace"):
    if hasattr(value, "model_dump"):
        value = value.model_dump()
    elif hasattr(value, "dict") and not isinstance(value, dict):
        value = value.dict()
    elif hasattr(value, "__dict__"):
        value = vars(value)

    if isinstance(value, dict):
        metric_data = value.get("metrics_data")
        if isinstance(metric_data, list):
            for metric in metric_data:
                if hasattr(metric, "model_dump"):
                    metric = metric.model_dump()
                elif hasattr(metric, "dict") and not isinstance(metric, dict):
                    metric = metric.dict()
                elif hasattr(metric, "__dict__"):
                    metric = vars(metric)
                if isinstance(metric, dict):
                    evaluation_results.append({
                        "test_case": test_case_number,
                        "span": path,
                        "metric": metric.get("name"),
                        "score": metric.get("score"),
                        "success": metric.get("success"),
                        "reason": metric.get("reason"),
                    })
        for key, child in value.items():
            collect_metric_data(child, test_case_number, f"{path}.{key}")
    elif isinstance(value, list):
        for child_index, child in enumerate(value):
            collect_metric_data(child, test_case_number, f"{path}[{child_index}]")

for index, golden in enumerate(agent_dataset.goldens[:10]):
    answer, actual_tools = run_agentic_evaluation_case(golden)

    test_case = LLMTestCase(
        input=golden.input,
        actual_output=answer,
        expected_output=golden.expected_output,
        tools_called=actual_tools or [],
        expected_tools=golden.expected_tools or [],
    )

    evaluation_result = evaluate(
        test_cases=[test_case],
        metrics=[task_completion_metric, tool_correctness_metric],
        error_config=ErrorConfig(ignore_errors=False),
        async_config=AsyncConfig(run_async=False),
    )

    collect_metric_data(evaluation_result, index + 1)
    time.sleep(60)

print("Task Completion + Tool Correctness evaluation completed.")
results_df = pd.DataFrame(evaluation_results)
display(results_df)

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

✨ You're running DeepEval's latest Task Completion Metric! (using openai/gpt-oss-20b, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=False)...

Output()

Prompt characters: 3066

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Task Completion            │ 0.90                  │ 100.00% | passed=1 | failed=0                 │ 1         │
│  Tool Correctness           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=689764;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.95s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Task Completion + Tool Correctness evaluation completed.


,test_case,span,metric,score,success,reason
0,1,trace.test_results[0],Task Completion,1.00,True,The system provided the requested status of or...
1,1,trace.test_results[0],Tool Correctness,1.00,True,[\n\t Tool Calling Reason: All expected tools ...
2,2,trace.test_results[0],Task Completion,0.60,True,The response correctly states the general 30‑d...
3,2,trace.test_results[0],Tool Correctness,1.00,True,[\n\t Tool Calling Reason: All expected tools ...
4,3,trace.test_results[0],Task Completion,0.30,False,The assistant only prompted the user for a new...
5,3,trace.test_results[0],Tool Correctness,1.00,True,[\n\t Tool Calling Reason: All expected tools ...
6,4,trace.test_results[0],Task Completion,0.95,True,The response directly addresses the user’s req...
7,4,trace.test_results[0],Tool Correctness,0.00,False,[\n\t Tool Calling Reason: Incomplete tool usa...
8,5,trace.test_results[0],Task Completion,0.40,False,The system only requested additional informati...
9,5,trace.test_results[0],Tool Correctness,0.00,False,[\n\t Tool Calling Reason: Incomplete tool usa...


### TaskCompletionMetric

TaskCompletionMetric checks whether an AI agent successfully completed the user’s task.

It focuses on the final outcome of the agent’s work.

If the agent’s response fully satisfies the user request, it passes.

If the agent gives only partial help, unclear help, or does not complete the requested task, it fails.

In [ ]:
import pandas as pd

agent_df = pd.read_csv("agentic_metrics_dataset.csv")

print("Agentic metrics dataset loaded.")
print("Total test cases:", len(agent_df))

display(agent_df.head())

This approach is used when QA does not have access to the live agent source code or tracing callbacks. In that case, actual_tools and expected_tools can be compared from logs/CSV.

If agent source code is available, ToolCorrectnessMetric can be implemented in DeepEval’s tracing style using @observe, Golden(expected_tools=...), EvaluationDataset, and evals_iterator(), where actual tool calls are captured automatically from the agent execution.

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
import time

tool_correctness_results = []

for row_index, row in agent_df.iterrows():
    task_completion_test_case = LLMTestCase(
        input=row["user_task"],
        actual_output=row["actual_output"]
    )

    print("Running:", row["test_id"])

    evaluate(
        test_cases=[task_completion_test_case],
        metrics=[task_completion_metric],
        async_config=AsyncConfig(run_async=False),
        error_config=ErrorConfig(ignore_errors=True)
    )

    actual_tool = str(row["actual_tools"]).strip()
    expected_tool = str(row["expected_tools"]).strip()

    tool_correctness_status = "PASS" if actual_tool == expected_tool else "FAIL"

    tool_correctness_results.append({
        "test_id": row["test_id"],
        "user_task": row["user_task"],
        "expected_tool": expected_tool,
        "actual_tool": actual_tool,
        "tool_correctness": tool_correctness_status
    })

    print("Expected tool:", expected_tool)
    print("Actual tool:", actual_tool)
    print("Tool Correctness:", tool_correctness_status)
    print("Completed:", row["test_id"])
    print("-" * 80)

    time.sleep(5)

tool_correctness_df = pd.DataFrame(tool_correctness_results)

display(tool_correctness_df)

print("Task Completion + Tool Correctness evaluation completed.")

StepEfficiencyMetric

StepEfficiencyMetric checks whether an AI agent completed the task using useful and necessary steps.

It focuses on the agent’s execution path.

If the agent uses direct and relevant steps, it passes.

If the agent uses unnecessary, repeated, or unrelated steps, it fails.

In [ ]:
step_efficiency_metric = StepEfficiencyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Step Efficiency metric created.")

In [ ]:
for golden in agent_dataset.evals_iterator(
    metrics=[step_efficiency_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = customer_support_agent(golden.input)

    time.sleep(5)


print("Agentic Step Efficiency evaluation completed.")

ToolCorrectnessMetric

ToolCorrectnessMetric checks whether an AI agent selected the correct tool for the given task.

It compares the tools actually used by the agent with the tools expected for that task.

If the agent calls the correct tool, it passes.

If the agent calls the wrong tool, misses a required tool, or uses an unnecessary tool, it fails.

In [ ]:
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval import evaluate

In [ ]:
tool_correctness_metric = ToolCorrectnessMetric(
    threshold=0.6
)

print("Tool Correctness metric created.")
# ToolCorrectnessMetric can compare expected tool and actual tool directly.

ArgumentCorrectnessMetric

ArgumentCorrectnessMetric checks whether an AI agent passed the correct arguments or input values into the selected tool.

It is used after checking tool correctness.

If the agent selects the correct tool but passes wrong, missing, or incomplete arguments, this metric can fail.

Example:
If the task is “Track order ORD123”, the agent should call the tracking tool with order_id = "ORD123".

If the agent calls the tracking tool without the order ID, the tool choice is correct, but the argument is wrong.

In [ ]:
argument_correctness_metric = ArgumentCorrectnessMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Argument Correctness metric created.")

In [ ]:
argument_test_cases = [
    LLMTestCase(
        input="Track order ORD123.",
        actual_output="Order ORD123 can be tracked using the tracking link.",
        tools_called=[
            ToolCall(
                name="track_order_tool",
                input_parameters={"order_id": "ORD123"}
            )
        ]
    ),
    LLMTestCase(
        input="Check refund policy for order ORD456.",
        actual_output="Order ORD456 is eligible for refund if it is within 15 days and unused.",
        tools_called=[
            ToolCall(
                name="refund_policy_tool",
                input_parameters={"order_id": "ORD456"}
            )
        ]
    ),
    LLMTestCase(
        input="Change delivery address for order ORD789.",
        actual_output="Delivery address for order ORD789 can be changed before shipment.",
        tools_called=[
            ToolCall(
                name="change_address_tool",
                input_parameters={"order_id": "ORD789"}
            )
        ]
    )
]

evaluate(
    test_cases=argument_test_cases,
    metrics=[argument_correctness_metric]
)

print("Argument Correctness evaluation completed.")

PlanQualityMetric

PlanQualityMetric checks whether an AI agent created a good plan before doing the task.

It focuses on the quality of the planned steps.

A good plan should be clear, logical, complete, and useful for completing the user’s task.

If the plan is missing important steps, has unnecessary steps, or is not useful for the task, this metric can fail.

In [ ]:
plan_quality_metric = PlanQualityMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Plan Quality metric created.")

In [ ]:
@observe()
def planned_customer_support_agent(user_task):
    plan = [
        "Identify the customer support request.",
        "Select the correct support tool.",
        "Use the tool result to answer the customer."
    ]

    if "track" in user_task.lower():
        answer = track_order_tool()

    elif "refund" in user_task.lower():
        answer = refund_policy_tool()

    elif "delivery address" in user_task.lower() or "address" in user_task.lower():
        answer = change_address_tool()

    else:
        answer = "I could not identify the correct support action."

    update_current_trace(
        input=user_task,
        output=answer,
        metadata={
            "plan": plan
        }
    )

    return answer

In [ ]:
for golden in agent_dataset.evals_iterator(
    metrics=[plan_quality_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = planned_customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Plan Quality evaluation completed.")

PlanAdherenceMetric

PlanAdherenceMetric checks whether an AI agent followed the plan it created.

It compares the planned steps with the actual steps taken by the agent.

If the agent follows the planned steps properly, it passes.

If the agent skips planned steps, does different steps, or goes away from the plan, this metric can fail.

In [ ]:
plan_adherence_metric = PlanAdherenceMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Plan Adherence metric created.")

In [ ]:
for golden in agent_dataset.evals_iterator(
    metrics=[plan_adherence_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = planned_customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Plan Adherence evaluation completed.")

### Trace inspection
Inspect the latest trace before removing any data. This identifies oversized or repeated fields while preserving the evidence needed for evaluation.

In [ ]:
print("Inspection cell started")
from deepeval.tracing import trace_manager

traces = trace_manager.get_all_traces_dict()

if not traces:
    print("No trace found.")
else:
    latest_trace = traces[-1]

    def inspect_trace(value, path="trace"):
        if isinstance(value, dict):
            for key, child in value.items():
                current_path = f"{path}.{key}"

                if key in {
                    "input",
                    "output",
                    "tools_called",
                    "expected_tools",
                    "metadata",
                }:
                    print(f"{current_path}: {len(str(child))} characters")
                    print(child)

                if isinstance(child, (dict, list)):
                    inspect_trace(child, current_path)

        elif isinstance(value, list):
            for index, child in enumerate(value):
                inspect_trace(child, f"{path}[{index}]")

    inspect_trace(latest_trace)

Inspection cell started
